In [1]:
import sqlite3

db_file = "/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/database/SMART.db"

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Get all user tables
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""")

tables = [row[0] for row in cursor.fetchall()]

for table in tables:
    print(f"\nTable: {table}")

    cursor.execute(f"PRAGMA table_info('{table}')")
    columns = cursor.fetchall()

    for col in columns:
        # col format:
        # (cid, name, type, notnull, dflt_value, pk)
        print(f"  {col[1]} ({col[2]})")

conn.close()


Table: ip_addresses
  id (CHAR(32))
  ip_address (TEXT)
  rdns_hostname (TEXT)
  asn (INTEGER)
  asn_org (TEXT)
  country_code (TEXT)
  country_rating (INTEGER)
  asn_rating (INTEGER)

Table: mail_system_ip_history
  id (CHAR(32))
  mail_system_id (CHAR(32))
  ip_address_id (CHAR(32))
  valid_from_run (CHAR(32))
  valid_to_run (CHAR(32))
  is_current (BOOLEAN)

Table: mail_systems
  id (CHAR(32))
  role (VARCHAR(9))
  software (TEXT)
  vendor (TEXT)
  vendor_country (TEXT)
  vendor_category (TEXT)
  vendor_country_rating (INTEGER)
  open_source_rating (INTEGER)
  vendor_category_rating (INTEGER)

Table: org_domain_history
  id (CHAR(32))
  organisation_id (CHAR(32))
  valid_from_run (CHAR(32))
  valid_to_run (CHAR(32))
  is_current (BOOLEAN)
  email_domain (TEXT)
  website_domain (TEXT)

Table: org_mail_system_history
  id (CHAR(32))
  organisation_id (CHAR(32))
  mail_system_id (CHAR(32))
  valid_from_run (CHAR(32))
  valid_to_run (CHAR(32))
  proxy_system_id (CHAR(32))
  is_current 

In [1]:
import pandas as pd
path = "/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/exports"
smtp = pd.read_csv(path+"/smtp.csv")
smtp

,mx_domain,smtp_banner,port,error
0,mx2.belwue.de,220 mail-in02.belwue.de ESMTP Postfix,25.0,NaN
1,mx3.belwue.de,220 mail-in03.belwue.de ESMTP Postfix,25.0,NaN
2,mx1.belwue.de,220 mail-in02.belwue.de ESMTP Postfix,25.0,NaN
3,mx4.belwue.de,220 mail-in04.belwue.de ESMTP Postfix,25.0,NaN
4,burg-halle.de,220 ******************************************...,25.0,NaN
...,...,...,...,...
2925,wels-gv-at.mail.eo.outlook.com,220 AM3PEPF0000A790.mail.protection.outlook.co...,25.0,NaN
2926,laakirchen-ooe-gv-at.mail.protection.outlook.com,220 VI4PEPF00000001.mail.protection.outlook.co...,25.0,NaN
2927,msx1.salzburg.gv.at,554-msx2.salzburg.gv.at,25.0,NaN
2928,mx2.hc1656-87.eu.iphmx.com,220 esa2.hc1656-87.eu.iphmx.com ESMTP,25.0,NaN


In [15]:
ignore = ["220", "Fri,", "Jun", "at", "2026", "ready", "19", "+0200", "+0000", "-"]

keywords = (
    smtp["smtp_banner"]
    .str.split()
    .explode()
    .loc[lambda s: ~s.isin(ignore)]
    .value_counts()
)
keywords

smtp_banner
ESMTP                                          2057
Postfix                                         527
Service                                         461
Microsoft                                       373
MAIL                                            373
                                               ... 
VI4PEPF00000001.mail.protection.outlook.com       1
[08DEC92A93255CD6]                                1
554-msx2.salzburg.gv.at                           1
esa2.hc1656-87.eu.iphmx.com                       1
esa1.hc1656-87.eu.iphmx.com                       1
Name: count, Length: 3100, dtype: int64

In [3]:
import re
import yaml
from dataclasses import dataclass
from typing import Any


@dataclass
class MatchResult:
    banner: str
    software: str | None = None
    vendor: str | None = None
    vendor_country: str | None = None
    vendor_category: str | None = None
    regex: str | None = None


def load_yaml(path: str) -> list[dict]:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def apply_rules(banners: list[str], rules: list[dict]) -> tuple[list[MatchResult], list[str]]:
    remaining = banners[:]
    results: list[MatchResult] = []

    for rule in rules:
        regex = re.compile(rule["regex"])

        still_remaining = []

        for banner in remaining:
            if regex.search(banner):
                results.append(
                    MatchResult(
                        banner=banner,
                        software=rule.get("software"),
                        vendor=rule.get("vendor"),
                        vendor_country=rule.get("vendor_country"),
                        vendor_category=rule.get("vendor_category"),
                        regex=rule["regex"],
                    )
                )
            else:
                still_remaining.append(banner)

        remaining = still_remaining

    return results, remaining

import pandas as pd
import re
from collections import Counter
from typing import List, Optional, Set


def extract_keywords(
    domains: List[str],
    remove_tlds: Optional[Set[str]] = None
) -> pd.DataFrame:
    """
    Extract keywords from domain list and return frequency table.
    
    Parameters:
    - domains: list of domain strings
    - remove_tlds: set like {".de", ".com"} to filter out TLD tokens
    
    Returns:
    - DataFrame with columns: keyword, count
    """

    counter = Counter()

    for domain in domains:
        domain = domain.lower().strip()

        # split by special characters
        parts = re.split(r"[.\-_/]", domain)

        for part in parts:
            part = part.strip()

            if not part:
                continue

            # filter TLDs if needed
            if remove_tlds and part in remove_tlds:
                continue

            # remove numeric-only or garbage tokens
            if part.isdigit():
                continue

            counter[part] += 1

    df = pd.DataFrame(counter.items(), columns=["keyword", "count"])
    df = df.sort_values("count", ascending=False).reset_index(drop=True)

    return df

In [18]:
pd.DataFrame(matches)

,banner,software,vendor,vendor_country,vendor_category,regex
0,220 mx-ber.mail-bz.de - NoSpamProxy ready,NoSpamProxy,Net at Work GmbH,DE,Security / Proxy (EU Vendor),(?i)nospamproxy
1,220 mx-hh.mail-bz.de - NoSpamProxy ready,NoSpamProxy,Net at Work GmbH,DE,Security / Proxy (EU Vendor),(?i)nospamproxy
2,220 dhpolpost.dhpol.de - NoSpamProxy ready,NoSpamProxy,Net at Work GmbH,DE,Security / Proxy (EU Vendor),(?i)nospamproxy
3,220 mx.bu-jordan.de - NoSpamProxy ready,NoSpamProxy,Net at Work GmbH,DE,Security / Proxy (EU Vendor),(?i)nospamproxy
4,220 mailin.campus-lb.net - NoSpamProxy ready,NoSpamProxy,Net at Work GmbH,DE,Security / Proxy (EU Vendor),(?i)nospamproxy
...,...,...,...,...,...,...
1172,220-pmg.amtruhland.local ESMTP Proxmox,Proxmox Mail Gateway,Proxmox Server Solutions GmbH,AT,EU-Mailsoftware-Anbieter,(?i)proxmox
1173,220-mx1.lychen.de ESMTP Proxmox,Proxmox Mail Gateway,Proxmox Server Solutions GmbH,AT,EU-Mailsoftware-Anbieter,(?i)proxmox
1174,220-mailgw.teterow.de ESMTP Proxmox,Proxmox Mail Gateway,Proxmox Server Solutions GmbH,AT,EU-Mailsoftware-Anbieter,(?i)proxmox
1175,220-mailgate.halberstadt.de ESMTP Proxmox,Proxmox Mail Gateway,Proxmox Server Solutions GmbH,AT,EU-Mailsoftware-Anbieter,(?i)proxmox


In [20]:
rules = load_yaml("/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/src/signatures_pipeline/signatures/smtp.yaml")

banners = smtp["smtp_banner"].dropna().tolist()

matches, unmatched = apply_rules(banners, rules)
print(f"Coverage: {len(matches)/len(banners)}")
unmatched

result = extract_keywords(
    unmatched,
    remove_tlds={"de", "mail", "mx"}
)
result

Coverage: 0.43608743979251574


,keyword,count
0,220 mail,268
1,de esmtp,145
2,de esmtp ready,145
3,com esmtp,100
4,ess,64
...,...,...
1985,220 hermes32,1
1986,220 hermes31,1
1987,220 kliniken,1
1988,(c) mailsheriff esmtp service ready to connect,1


In [2]:
import pandas as pd
path = "/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/exports"
asn = pd.read_csv(path+"/asn.csv")
combiner = pd.read_csv(path+"/combiner.csv")
domain = pd.read_csv(path+"/domain.csv")
imap = pd.read_csv(path+"/imap.csv")
ip = pd.read_csv(path+"/ip.csv")
mx = pd.read_csv(path+"/mx.csv")
smtp = pd.read_csv(path+"/smtp.csv")
ptr = pd.read_csv(path+"/ptr.csv")
domain

,id,organisation_id,valid_from_run,valid_to_run,is_current,email_domain,website_domain
0,276a3ae08ea045a9977a9345b110f5ba,a87f2eaab2a344afa905735b429fc142,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,dhbw-loerrach.de
1,cf8ed1372a90485489eb54b4d7d58292,3117434b47164a4a9833e9d2feff9fad,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,burg-halle.de
2,c491c1cd523842e7a2070f030a6e4832,3f0585cf1d844cb3894c317102413b48,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,businessschool-berlin.de
3,5590cd997d3f4724ad22ce17cc170a32,1a77dd9022f3459fa50cea34bc757033,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,cvjm-hochschule.de
4,ed694e4f09a14f52b913ab659bce93d4,80a41cd1328d4cbdac16426cbc9cb63a,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,tuhh.de
...,...,...,...,...,...,...,...
5020,bee3576183e348cc8cc897a1df4615b0,478a7d7e5446482db895bbe19fd649b7,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,bs.ch
5021,a04fd7fdcedb4c519ac8855b221a5675,f276104dde32449cbbb1cea70f89ac96,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,lausanne.ch
5022,b60a17c316754e9ca5420be3b10401a5,35dae49712034959b9fe929ac64b78ff,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,stadt-schaffhausen.ch
5023,70dff7b817694672ba802965065d3ab7,c9bb002d4f2b4a72bad9e1fa043083fd,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,buelach.ch


In [3]:
domain

,id,organisation_id,valid_from_run,valid_to_run,is_current,email_domain,website_domain
0,276a3ae08ea045a9977a9345b110f5ba,a87f2eaab2a344afa905735b429fc142,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,dhbw-loerrach.de
1,cf8ed1372a90485489eb54b4d7d58292,3117434b47164a4a9833e9d2feff9fad,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,burg-halle.de
2,c491c1cd523842e7a2070f030a6e4832,3f0585cf1d844cb3894c317102413b48,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,businessschool-berlin.de
3,5590cd997d3f4724ad22ce17cc170a32,1a77dd9022f3459fa50cea34bc757033,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,cvjm-hochschule.de
4,ed694e4f09a14f52b913ab659bce93d4,80a41cd1328d4cbdac16426cbc9cb63a,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,tuhh.de
...,...,...,...,...,...,...,...
5020,bee3576183e348cc8cc897a1df4615b0,478a7d7e5446482db895bbe19fd649b7,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,bs.ch
5021,a04fd7fdcedb4c519ac8855b221a5675,f276104dde32449cbbb1cea70f89ac96,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,lausanne.ch
5022,b60a17c316754e9ca5420be3b10401a5,35dae49712034959b9fe929ac64b78ff,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,stadt-schaffhausen.ch
5023,70dff7b817694672ba802965065d3ab7,c9bb002d4f2b4a72bad9e1fa043083fd,6d9c5553cdfb46d4844940801a9559f4,NaN,1,NaN,buelach.ch


In [7]:
ip

,mx_domain,ip,error
0,mx2.belwue.de,2001:7c0:0:76::2,NaN
1,mx2.belwue.de,129.143.76.2,NaN
2,mx3.belwue.de,2001:7c0:0:76::3,NaN
3,mx3.belwue.de,129.143.76.3,NaN
4,mx1.belwue.de,2001:7c0:0:76::1,NaN
...,...,...,...
6196,msx1.salzburg.gv.at,193.41.228.254,NaN
6197,mx2.hc1656-87.eu.iphmx.com,23.90.122.224,NaN
6198,mx2.hc1656-87.eu.iphmx.com,23.90.123.49,NaN
6199,mx1.hc1656-87.eu.iphmx.com,23.90.123.49,NaN


In [25]:
smtp

,mx_domain,smtp_banner,port,error
0,mx2.belwue.de,220 mail-in02.belwue.de ESMTP Postfix,25.0,NaN
1,mx3.belwue.de,220 mail-in03.belwue.de ESMTP Postfix,25.0,NaN
2,mx1.belwue.de,220 mail-in02.belwue.de ESMTP Postfix,25.0,NaN
3,mx4.belwue.de,220 mail-in04.belwue.de ESMTP Postfix,25.0,NaN
4,burg-halle.de,220 ******************************************...,25.0,NaN
...,...,...,...,...
2925,wels-gv-at.mail.eo.outlook.com,220 AM3PEPF0000A790.mail.protection.outlook.co...,25.0,NaN
2926,laakirchen-ooe-gv-at.mail.protection.outlook.com,220 VI4PEPF00000001.mail.protection.outlook.co...,25.0,NaN
2927,msx1.salzburg.gv.at,554-msx2.salzburg.gv.at,25.0,NaN
2928,mx2.hc1656-87.eu.iphmx.com,220 esa2.hc1656-87.eu.iphmx.com ESMTP,25.0,NaN


In [32]:
start = mx[mx["error"].isna()].drop_duplicates("domain").drop(columns="error")

start = start.merge(
    ip.drop_duplicates(subset=["mx_domain"]),
    on="mx_domain",
    how="left",
    validate="many_to_one",   # viele Domains -> eine IP-Zeile pro mx_domain
).drop(columns="error")

start = start.merge(
    ptr.drop_duplicates(subset=["ip"]),
    on="ip",
    how="left",
    validate="many_to_one",   # viele Domains -> eine PTR-Zeile pro IP
).drop(columns="error")

start = start.merge(
    asn,
    on="ip",
    how="left",
    validate="many_to_one",   # viele Domains -> eine ASN-Zeile pro IP
).drop(columns=["ip", "prefix", "error", "asn"])

start = start.merge(
    imap.rename(columns={"banner": "imap_banner"}),
    on="domain",
    how="left",
    validate="one_to_one",    # domain sollte links und rechts eindeutig sein
).drop(columns=["error", "port"])

start = start.merge(
    smtp,
    on="mx_domain",
    how="left",
    validate="many_to_one",   # viele Domains -> eine SMTP-Zeile pro mx_domain
).drop(columns=["error", "port"])
start.to_csv("/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/exports/combined_scanning_results.csv")

In [ ]:
start = mx[mx["error"].isna()].drop_duplicates("domain").drop(columns="error")
start = start.merge(ip.drop_duplicates("mx_domain"), on="mx_domain", how="left").drop(columns="error")
start = start.merge(ptr, on="ip", how="left").drop(columns="error")
start = start.merge(asn, on="ip", how="left").drop(columns=["ip", "prefix", "error", "asn"])
start = start.merge(imap.rename(columns={"banner":"imap_banner"}), on="domain", how="left").drop(columns=["error", "port"])
start = start.merge(smtp, on="mx_domain", how="left").drop(columns=["error", "port"])
start

,domain,mx_domain,ptr,owner,country,imap_host,imap_banner,smtp_banner
0,dhbw-loerrach.de,mx2.belwue.de,mail-in02.belwue.de,"BELWUE - Universitaet Stuttgart, DE",DE,imap.dhbw-loerrach.de,* OK IMAP4 ready,220 mail-in02.belwue.de ESMTP Postfix
1,dhbw-loerrach.de,mx2.belwue.de,mail-in02.belwue.de,"BELWUE - Universitaet Stuttgart, DE",DE,imap.dhbw-loerrach.de,* OK IMAP4 ready,220 mail-in02.belwue.de ESMTP Postfix
2,burg-halle.de,burg-halle.de,burg-halle.de,"HLKOMM - HL komm Telekommunikations GmbH, DE",DE,NaN,NaN,220 ******************************************...
3,businessschool-berlin.de,mx-ber.mail-bz.de,mx-ber.mail-bz.de,"VERSATEL - 1&1 Versatel GmbH, DE",DE,NaN,NaN,220 mx-ber.mail-bz.de - NoSpamProxy ready
4,cvjm-hochschule.de,mx1.serve-me.de,mx1.serve-me.de,"HETZNER-AS - Hetzner Online GmbH, DE",DE,NaN,NaN,220 mx1.serve-me.de ESMTP
...,...,...,...,...,...,...,...,...
7163,klingenbach.bgld.gv.at,smtp.asp-bgld.at,smtp.asp-bgld.at,"KABSI-AS - kabelplus GmbH, AT",AT,NaN,NaN,"220 smtp.asp-bgld.at ESMTP Smtpd; Fri, 19 Jun ..."
7164,ktn.gde.at,mx2.hc1656-87.eu.iphmx.com,esa2.hc1656-87.eu.iphmx.com,AS-IRONP-VEGA - Cisco Systems Ironport Divisio...,US,NaN,NaN,220 esa2.hc1656-87.eu.iphmx.com ESMTP
7165,ktn.gde.at,mx2.hc1656-87.eu.iphmx.com,esa1.hc1656-87.eu.iphmx.com,AS-IRONP-VEGA - Cisco Systems Ironport Divisio...,US,NaN,NaN,220 esa2.hc1656-87.eu.iphmx.com ESMTP
7166,kittsee.bgld.gv.at,smtp.asp-bgld.at,smtp.asp-bgld.at,"KABSI-AS - kabelplus GmbH, AT",AT,NaN,NaN,"220 smtp.asp-bgld.at ESMTP Smtpd; Fri, 19 Jun ..."


In [2]:
import re
import yaml
from dataclasses import dataclass
from typing import Any


@dataclass
class MatchResult:
    banner: str
    software: str | None = None
    vendor: str | None = None
    vendor_country: str | None = None
    vendor_category: str | None = None
    regex: str | None = None


def load_yaml(path: str) -> list[dict]:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def apply_rules(banners: list[str], rules: list[dict]) -> tuple[list[MatchResult], list[str]]:
    remaining = banners[:]
    results: list[MatchResult] = []

    for rule in rules:
        regex = re.compile(rule["regex"])

        still_remaining = []

        for banner in remaining:
            if regex.search(banner):
                results.append(
                    MatchResult(
                        banner=banner,
                        software=rule.get("software"),
                        vendor=rule.get("vendor"),
                        vendor_country=rule.get("vendor_country"),
                        vendor_category=rule.get("vendor_category"),
                        regex=rule["regex"],
                    )
                )
            else:
                still_remaining.append(banner)

        remaining = still_remaining

    return results, remaining

In [ ]:
rules = load_yaml("/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/src/signatures_pipeline/signatures/mx.yaml")

banners = mx["mx_domain"].dropna().tolist()

matches, unmatched = apply_rules(banners, rules)
print(f"Coverage: {len(matches)/len(banners)}")
unmatched

result = extract_keywords(
    unmatched,
    remove_tlds={"de", "com", "at", "mail", "mx", "ch", "mx1", "mx2","net"}
)
result

Coverage: 0.49255014326647567


,keyword,count
0,rlp,156
1,mail2,132
2,uni,128
3,mx01,127
4,in,111
...,...,...
2232,bwl,1
2233,hermes32,1
2234,hermes31,1
2235,email2,1


In [42]:
import pandas as pd
import re
from collections import Counter
from typing import List, Optional, Set


def extract_keywords(
    domains: List[str],
    remove_tlds: Optional[Set[str]] = None
) -> pd.DataFrame:
    """
    Extract keywords from domain list and return frequency table.
    
    Parameters:
    - domains: list of domain strings
    - remove_tlds: set like {".de", ".com"} to filter out TLD tokens
    
    Returns:
    - DataFrame with columns: keyword, count
    """

    counter = Counter()

    for domain in domains:
        domain = domain.lower().strip()

        # split by special characters
        parts = re.split(r"[.\-_/]", domain)

        for part in parts:
            part = part.strip()

            if not part:
                continue

            # filter TLDs if needed
            if remove_tlds and part in remove_tlds:
                continue

            # remove numeric-only or garbage tokens
            if part.isdigit():
                continue

            counter[part] += 1

    df = pd.DataFrame(counter.items(), columns=["keyword", "count"])
    df = df.sort_values("count", ascending=False).reset_index(drop=True)

    return df

result = extract_keywords(
    unmatched,
    remove_tlds={"de", "com", "at", "mail", "mx", "ch", }
)
result

,keyword,count
0,mx1,201
1,mx2,189
2,net,184
3,rlp,156
4,mail2,132
...,...,...
2297,ooe,1
2298,braunau,1
2299,relay7m,1
2300,relay7v,1


In [52]:
remaining = pd.DataFrame(unmatched, columns=["mx_domain"])
remaining["mx_domain"].value_counts()

mx_domain
sslmail.verwaltungsportal.de             65
mail.bayern.de                           64
mxlb.ispgateway.de                       45
mx-in.kommunale.it                       44
mailmx0001.rlp.de                        40
                                         ..
braunau-ooe-gv-at.gate.seppmail.cloud     1
srv-gw-mx1.magibk.at                      1
srv-gw-mx2.magibk.at                      1
wels-gv-at.mail.eo.outlook.com            1
msx1.salzburg.gv.at                       1
Name: count, Length: 2204, dtype: int64

In [15]:
matches

[MatchResult(banner='* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE LITERAL+ STARTTLS LOGINDISABLED] Dovecot (Debian) ready.', software='Dovecot', vendor='Open Source', vendor_country='FI', vendor_category='Mail Server', regex='(?i)dovecot'),
 MatchResult(banner='* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE LITERAL+ AUTH=PLAIN AUTH=GSSAPI] Dovecot (Debian) ready.', software='Dovecot', vendor='Open Source', vendor_country='FI', vendor_category='Mail Server', regex='(?i)dovecot'),
 MatchResult(banner='* OK [CAPABILITY IMAP4rev1 LITERAL+ SASL-IR LOGIN-REFERRALS ID ENABLE IDLE AUTH=PLAIN] Dovecot ready.', software='Dovecot', vendor='Open Source', vendor_country='FI', vendor_category='Mail Server', regex='(?i)dovecot'),
 MatchResult(banner='* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE LITERAL+ STARTTLS LOGINDISABLED] Dovecot ready.', software='Dovecot', vendor='Open Source', vendor_country='FI', vendor_category='Mail Server', rege

In [12]:
imap["banner"].dropna().value_counts()

banner
* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE LITERAL+ STARTTLS AUTH=PLAIN AUTH=LOGIN] Dovecot ready.                                                                                   118
* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE XLIST LITERAL+ STARTTLS AUTH=PLAIN AUTH=LOGIN] kasserver.com mailserver ready.                                                            105
* OK [CAPABILITY IMAP4rev1 LOGIN-REFERRALS ID ENABLE IDLE SASL-IR LITERAL+ STARTTLS AUTH=PLAIN AUTH=LOGIN AUTH=DIGEST-MD5 AUTH=CRAM-MD5] Dovecot ready.                                                      69
* OK The Microsoft Exchange IMAP4 service is ready.                                                                                                                                                          67
* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFERRALS ID ENABLE IDLE LITERAL+ STARTTLS AUTH=PLAIN AUTH=LOGIN AUTH=CRAM-MD5] Mailserver ready.                       

In [4]:
combiner_mx = combiner.merge(mx, on="domain", how="left").drop(columns=["error"])
combiner_mx
combiner_mx_ip = combiner_mx.merge(ip.drop(columns=["error"]), on="mx_domain")
combiner_mx_ip
combiner_mx_ip_asn = combiner_mx_ip.merge(asn.drop(columns=["asn", "prefix", "error"]), on="ip")
combiner_mx_ip_asn
combiner_mx_ip_asn_smtp = combiner_mx_ip_asn.merge(smtp.drop(columns=["port", "error"]), on="mx_domain")
combiner_mx_ip_asn_smtp
combiner_mx_ip_asn_smtp_imap = combiner_mx_ip_asn_smtp.merge(imap.drop(columns=["port", "error"]), on="domain")
combiner_mx_ip_asn_smtp_imap
# domain_combiner_mx = domain.merge(combiner_mx, left_on="email", right_on="domain", how="left")
# domain_combiner_mx = domain.merge(combiner_mx, left_on="email", right_on="domain", how="left")
# domain_combiner_mx
# combiner_mx_domain_ip = combiner_mx_domain.merge(ip.drop(columns=["error"]), on="mx_domain", how="outer")
# combiner_mx_domain_ip

,domain,mx_domain,ip,owner,country,smtp_banner,imap_host,banner
0,dhbw-loerrach.de,mx3.belwue.de,2001:7c0:0:76::3,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in03.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
1,dhbw-loerrach.de,mx3.belwue.de,129.143.76.3,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in03.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
2,dhbw-loerrach.de,mx2.belwue.de,2001:7c0:0:76::2,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in02.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
3,dhbw-loerrach.de,mx2.belwue.de,129.143.76.2,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in02.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
4,dhbw-loerrach.de,mx1.belwue.de,2001:7c0:0:76::2,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in02.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
...,...,...,...,...,...,...,...,...
13282,zermatt.ch,zermatt-ch.mail.protection.outlook.com,2a01:111:f403:ca3c::1,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 GV1PEPF000006FB.mail.protection.outlook.co...,NaN,NaN
13283,zermatt.ch,zermatt-ch.mail.protection.outlook.com,52.101.185.12,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 GV1PEPF000006FB.mail.protection.outlook.co...,NaN,NaN
13284,zermatt.ch,zermatt-ch.mail.protection.outlook.com,52.101.185.13,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 GV1PEPF000006FB.mail.protection.outlook.co...,NaN,NaN
13285,zermatt.ch,zermatt-ch.mail.protection.outlook.com,52.101.187.1,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 GV1PEPF000006FB.mail.protection.outlook.co...,NaN,NaN


In [ ]:
temp = combiner_mx_ip_asn_smtp_imap.drop_duplicates(subset=["domain", "owner", "country"])
temp[temp["domain"].duplicated(keep=False)]

,domain,mx_domain,ip,owner,country,smtp_banner,imap_host,banner
27,university-of-labour.de,universityoflabour-de02ec.mail.protection.outl...,2a01:111:f403:ca2d::1,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 FR2PEPF000004F0.mail.protection.outlook.co...,imap.university-of-labour.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
31,university-of-labour.de,universityoflabour-de02ec.mail.protection.outl...,52.101.170.1,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 FR2PEPF000004F0.mail.protection.outlook.co...,imap.university-of-labour.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
39,srh-university.de,srhuniversity-de0i.mail.protection.outlook.com,2a01:111:f403:ca04::9,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 DU6PEPF0000B620.mail.protection.outlook.co...,NaN,NaN
43,srh-university.de,srhuniversity-de0i.mail.protection.outlook.com,52.101.68.39,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 DU6PEPF0000B620.mail.protection.outlook.co...,NaN,NaN
51,rptu.de,mailgate1.uni-kl.de,2001:638:208:120::208,DFN - Verein zur Foerderung eines Deutschen Fo...,DE,220 mailgw2.uni-kl.de ESMTP Sendmail 8.18.1/8....,mail.rptu.de,* OK mail1.uni-kl.de CommuniGate Pro IMAP Serv...
...,...,...,...,...,...,...,...,...
13258,steyr.gv.at,steyr-gv-at.mail.protection.outlook.com,52.101.68.8,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 AM2PEPF0001C717.mail.protection.outlook.co...,NaN,NaN
13262,laakirchen.ooe.gv.at,laakirchen-ooe-gv-at.mail.protection.outlook.com,2a01:111:f403:ca6e::3,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 VI6PEPF00000005.mail.protection.outlook.co...,NaN,NaN
13266,laakirchen.ooe.gv.at,laakirchen-ooe-gv-at.mail.protection.outlook.com,40.107.180.3,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 VI6PEPF00000005.mail.protection.outlook.co...,NaN,NaN
13279,zermatt.ch,zermatt-ch.mail.protection.outlook.com,2a01:111:f403:ca3c::2,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 GV1PEPF000006FB.mail.protection.outlook.co...,NaN,NaN


In [20]:
domain_email = domain[["name","email", "website_domain"]].merge(temp.drop_duplicates(subset="domain"), left_on="email", right_on="domain", how="left")

matched = domain_email[domain_email["domain"].notna()]

unmatched = (
    domain_email[domain_email["domain"].isna()]
    .drop(columns=temp.columns, errors="ignore")  # remove empty merge columns
    .merge(temp, left_on="website_domain", right_on="domain", how="left")
)

final = pd.concat([matched, unmatched], ignore_index=True).drop_duplicates("name")
final

,name,email,website_domain,domain,mx_domain,ip,owner,country,smtp_banner,imap_host,banner
0,CVJM-Hochschule,cvjm.de,cvjm-hochschule.de,cvjm.de,cvjm-de.mail.protection.outlook.com,2a01:111:f403:ca09::5,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 DU2PEPF00028D01.mail.protection.outlook.co...,imap.cvjm.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
1,BSBI School of Business and Innovation,berlinsbi.com,berlinsbi.com,berlinsbi.com,mx2.hc6443-94.iphmx.com,68.232.130.72,IRONPORT-SYSTEMS-INC - Cisco Systems Ironport ...,US,220 esa.hc6443-94.iphmx.com ESMTP,NaN,NaN
2,University of Labour,academy-of-labour.de,university-of-labour.de,academy-of-labour.de,academyoflabour-de01ie.mail.protection.outlook...,2a01:111:f403:ca2d::2,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 FR3PEPF00000484.mail.protection.outlook.co...,imap.academy-of-labour.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
3,Charlotte Fresenius Hochschule,hs-fresenius.de,charlotte-fresenius-uni.de,hs-fresenius.de,hsfresenius-de0e.mail.protection.outlook.com,52.101.73.153,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 DU2PEPF0001E9C1.mail.protection.outlook.co...,mail.hs-fresenius.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
4,Collegium Orientale,bistum-eichstaett.de,collegium-orientale.de,bistum-eichstaett.de,mx02.dioezesennetz.de,141.78.102.98,"ASN-TSI-IPLS - Telekom Deutschland GmbH, DE",DE,554-skibayi02.kirche-bayern.de,imap.bistum-eichstaett.de,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
...,...,...,...,...,...,...,...,...,...,...,...
5456,Zürich,NaN,stadt-zuerich.ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5457,Basel,NaN,bs.ch,bs.ch,mail10.swisscom.com,138.188.176.225,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH,220 mail.swisscom.com ESMTP Service Swisscom (...,NaN,NaN
5458,Schaffhausen,NaN,stadt-schaffhausen.ch,stadt-schaffhausen.ch,miraculix2.ksd.ch,195.246.245.126,"KSD - Informatik Schaffhausen (ITSH), CH",CH,554-miraculix2.ksd.ch,NaN,NaN
5459,Bülach,NaN,buelach.ch,buelach.ch,mailprotect3.itexacloud.ch,213.221.234.85,"QUICKLINE - Quickline AG, CH",CH,220 mailprotect3.itexacloud.ch,NaN,NaN


In [22]:
final[(final.website_domain == final.domain) & (final.email != final.domain)]

,name,email,website_domain,domain,mx_domain,ip,owner,country,smtp_banner,imap_host,banner
2398,Duale Hochschule Baden-Württemberg Lörrach,NaN,dhbw-loerrach.de,dhbw-loerrach.de,mx3.belwue.de,2001:7c0:0:76::3,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in03.belwue.de ESMTP Postfix,imap.dhbw-loerrach.de,* OK IMAP4 ready
2399,Burg Giebichenstein Kunsthochschule Halle,NaN,burg-halle.de,burg-halle.de,burg-halle.de,212.122.56.1,"HLKOMM - HL komm Telekommunikations GmbH, DE",DE,220 ******************************************...,NaN,NaN
2400,BSP Business & Law School,NaN,businessschool-berlin.de,businessschool-berlin.de,mx-ber.mail-bz.de,62.214.51.13,"VERSATEL - 1&1 Versatel GmbH, DE",DE,220 mx-ber.mail-bz.de - NoSpamProxy ready,NaN,NaN
2401,Technische Universität Hamburg,NaN,tuhh.de,tuhh.de,smtp3.rz.tu-harburg.de,2001:638:702:20aa::205:37,DFN - Verein zur Foerderung eines Deutschen Fo...,DE,220-smtp3.rz.tu-harburg.de ESMTP Postfix (Debian),mail.tuhh.de,* OK [CAPABILITY IMAP4rev1 LITERAL+ ID ENABLE ...
2402,SRH University Campus Fürth,NaN,srh-university.de,srh-university.de,srhuniversity-de0i.mail.protection.outlook.com,2a01:111:f403:ca04::9,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 DU6PEPF0000B620.mail.protection.outlook.co...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5452,Glarus Nord,NaN,glarus-nord.ch,glarus-nord.ch,ip17.gl.ch,217.192.170.97,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH,554-mx0.gl.ch,NaN,NaN
5453,Genf,NaN,geneve.ch,geneve.ch,geneve-ch.mail.protection.outlook.com,2a01:111:f403:ca3c::,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 ZR1PEPF0000077C.mail.protection.outlook.co...,NaN,NaN
5457,Basel,NaN,bs.ch,bs.ch,mail10.swisscom.com,138.188.176.225,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH,220 mail.swisscom.com ESMTP Service Swisscom (...,NaN,NaN
5458,Schaffhausen,NaN,stadt-schaffhausen.ch,stadt-schaffhausen.ch,miraculix2.ksd.ch,195.246.245.126,"KSD - Informatik Schaffhausen (ITSH), CH",CH,554-miraculix2.ksd.ch,NaN,NaN


In [19]:
final[final["name"].duplicated(keep=False)]

,name,email,website_domain,domain,mx_domain,ip,owner,country,smtp_banner,imap_host,banner
239,Pädagogische Hochschule Freiburg,unifr.ch,hepfr.ch,unifr.ch,mxa-00626501.gslb.pphosted.com,66.159.233.182,"PROOFPOINT-ASN-EU - Proofpoint, Inc., US",US,"220 mx07-00626501.pphosted.com ESMTP Wed, 10 J...",NaN,NaN
434,Krankenhaus Barmherzige Brüder,barmherzige-regensburg.de,barmherzige-regensburg.de,barmherzige-regensburg.de,mx04.hornetsecurity.com,94.100.136.7,"SSERV-AS - kyberio GmbH, DE",DE,220 mx-gate44-hz2.hornetsecurity.com,NaN,NaN
458,Martin-Luther-Krankenhaus,klinikum-bochum.de,martin-luther-krankenhaus-bo.de,klinikum-bochum.de,mx02.hornetsecurity.com,94.100.136.8,"SSERV-AS - kyberio GmbH, DE",DE,220 mx-gate45-hz2.hornetsecurity.com,NaN,NaN
614,Martini-Klinik,uke.de,martini-klinik.de,uke.de,iplexus2.uke.uni-hamburg.de,134.100.102.33,DFN - Verein zur Foerderung eines Deutschen Fo...,DE,554-iplexus.uke.uni-hamburg.de,imap.uke.de,* OK The Microsoft Exchange IMAP4 service is r...
952,Bundespatentgericht,bpatg.bund.de,bundespatentgericht.de,bpatg.bund.de,mx1.bund.de,77.87.224.131,BSI-AS - Bundesministerium des Innern und fuer...,DE,220 mx1.bund.de ESMTP,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5453,Genf,NaN,geneve.ch,geneve.ch,geneve-ch.mail.protection.outlook.com,2a01:111:f403:ca3c::,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 ZR1PEPF0000077C.mail.protection.outlook.co...,NaN,NaN
5454,Genf,NaN,geneve.ch,geneve.ch,geneve-ch.mail.protection.outlook.com,52.101.187.1,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,US,220 ZR1PEPF0000077C.mail.protection.outlook.co...,NaN,NaN
5459,Bülach,NaN,buelach.ch,buelach.ch,mailprotect3.itexacloud.ch,213.221.234.85,"QUICKLINE - Quickline AG, CH",CH,220 mailprotect3.itexacloud.ch,NaN,NaN
5460,Bülach,NaN,buelach.ch,buelach.ch,buelach-ch.mail.protection.outlook.com,2a01:111:f403:ca04::4,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 AMS1EPF0000004A.mail.protection.outlook.co...,NaN,NaN


In [15]:
domain_email[domain_email.email.isna()].merge(temp, left_on="email", right_on="domain", how="left")

,name,email,website_domain,domain_x,mx_domain_x,ip_x,owner_x,country_x,smtp_banner_x,imap_host_x,banner_x,domain_y,mx_domain_y,ip_y,owner_y,country_y,smtp_banner_y,imap_host_y,banner_y
0,Duale Hochschule Baden-Württemberg Lörrach,NaN,dhbw-loerrach.de,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Burg Giebichenstein Kunsthochschule Halle,NaN,burg-halle.de,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BSP Business & Law School,NaN,businessschool-berlin.de,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Technische Universität Hamburg,NaN,tuhh.de,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRH University Campus Fürth,NaN,srh-university.de,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2603,Zürich,NaN,stadt-zuerich.ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2604,Basel,NaN,bs.ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2605,Schaffhausen,NaN,stadt-schaffhausen.ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2606,Bülach,NaN,buelach.ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
domain_mx = pd.merge(domain[["name", "website_domain"]], mx, on="website_domain")
domain_mx

,name,website_domain,mx_domain,error
0,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx2.belwue.de,NaN
1,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx3.belwue.de,NaN
2,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx4.belwue.de,NaN
3,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx1.belwue.de,NaN
4,Burg Giebichenstein Kunsthochschule Halle,burg-halle.de,burg-halle.de,NaN
...,...,...,...,...
8938,Schaffhausen,stadt-schaffhausen.ch,miraculix2.ksd.ch,NaN
8939,Schaffhausen,stadt-schaffhausen.ch,miraculix1.ksd.ch,NaN
8940,Bülach,buelach.ch,buelach-ch.mail.protection.outlook.com,NaN
8941,Bülach,buelach.ch,mailprotect3.itexacloud.ch,NaN


In [10]:
import re

MX_PROVIDER_RULES = [
    (re.compile(r"\b(?:mx\d*\.)?hornetsecurity\."), "Hornetsecurity"),
    (re.compile(r"\bantispameurope\."), "Hornetsecurity"),  # ehemalige Marke
    (re.compile(r"\boutlook\.com$"), "Microsoft 365"),
    (re.compile(r"\bprotection\.outlook\.com$"), "Microsoft 365"),
    (re.compile(r"\bionos\."), "IONOS"),
    (re.compile(r"\bbelwue\."), "BelWü"),
    (re.compile(r"\bkvnb\."), "KVNB"),
]

def identify_provider(mx_domain: str) -> str | None:
    mx_domain = mx_domain.lower()

    for pattern, provider in MX_PROVIDER_RULES:
        if pattern.search(mx_domain):
            return provider

    return None

In [13]:
domain_mx["mx_provider"] = (
    domain_mx["mx_domain"]
    .fillna("")
    .apply(identify_provider)
)
domain_mx.drop_duplicates(subset="name", inplace=True)

In [14]:
domain_mx

,name,website_domain,mx_domain,error,mx_provider
0,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx2.belwue.de,NaN,BelWü
4,Burg Giebichenstein Kunsthochschule Halle,burg-halle.de,burg-halle.de,NaN,NaN
5,BSP Business & Law School,businessschool-berlin.de,mx-hh.mail-bz.de,NaN,NaN
7,CVJM-Hochschule,cvjm-hochschule.de,mx1.serve-me.de,NaN,NaN
9,Technische Universität Hamburg,tuhh.de,smtp3.rz.tu-harburg.de,NaN,NaN
...,...,...,...,...,...
8934,Basel,bs.ch,mail.swisscom.com,NaN,NaN
8937,Lausanne,lausanne.ch,mxvdl.lausanne.ch,NaN,NaN
8938,Schaffhausen,stadt-schaffhausen.ch,miraculix2.ksd.ch,NaN,NaN
8940,Bülach,buelach.ch,buelach-ch.mail.protection.outlook.com,NaN,Microsoft 365


In [27]:
domain_ip = pd.merge(domain_mx,  ip[["mx_domain","ip"]], on="mx_domain")
domain_asn = pd.merge(domain_ip, asn[["ip", "owner", "country"]], on="ip", how="left")
domain_asn = domain_asn.drop(columns=["error", "ip"]).drop_duplicates(subset="name")
domain_asn

,name,website_domain,mx_domain,mx_provider,owner,country
0,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx2.belwue.de,BelWü,"BELWUE - Universitaet Stuttgart, DE",DE
2,Burg Giebichenstein Kunsthochschule Halle,burg-halle.de,burg-halle.de,NaN,"HLKOMM - HL komm Telekommunikations GmbH, DE",DE
3,BSP Business & Law School,businessschool-berlin.de,mx-hh.mail-bz.de,NaN,"VERSATEL - 1&1 Versatel GmbH, DE",DE
4,CVJM-Hochschule,cvjm-hochschule.de,mx1.serve-me.de,NaN,"HETZNER-AS - Hetzner Online GmbH, DE",DE
5,Technische Universität Hamburg,tuhh.de,smtp3.rz.tu-harburg.de,NaN,DFN - Verein zur Foerderung eines Deutschen Fo...,DE
...,...,...,...,...,...,...
8456,Basel,bs.ch,mail.swisscom.com,NaN,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH
8458,Lausanne,lausanne.ch,mxvdl.lausanne.ch,NaN,SIL-CITYCABLE-AS - Services Industriels de Lau...,CH
8459,Schaffhausen,stadt-schaffhausen.ch,miraculix2.ksd.ch,NaN,"KSD - Informatik Schaffhausen (ITSH), CH",CH
8460,Bülach,buelach.ch,buelach-ch.mail.protection.outlook.com,Microsoft 365,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB


In [30]:
domain_smtp = pd.merge(domain_asn, smtp[["mx_domain", "smtp_banner"]], on="mx_domain")
domain_smtp

,name,website_domain,mx_domain,mx_provider,owner,country,smtp_banner
0,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx2.belwue.de,BelWü,"BELWUE - Universitaet Stuttgart, DE",DE,220 mail-in02.belwue.de ESMTP Postfix
1,Burg Giebichenstein Kunsthochschule Halle,burg-halle.de,burg-halle.de,NaN,"HLKOMM - HL komm Telekommunikations GmbH, DE",DE,220 ******************************************...
2,BSP Business & Law School,businessschool-berlin.de,mx-hh.mail-bz.de,NaN,"VERSATEL - 1&1 Versatel GmbH, DE",DE,220 mx-hh.mail-bz.de - NoSpamProxy ready
3,CVJM-Hochschule,cvjm-hochschule.de,mx1.serve-me.de,NaN,"HETZNER-AS - Hetzner Online GmbH, DE",DE,220 mx1.serve-me.de ESMTP
4,Technische Universität Hamburg,tuhh.de,smtp3.rz.tu-harburg.de,NaN,DFN - Verein zur Foerderung eines Deutschen Fo...,DE,220-smtp3.rz.tu-harburg.de ESMTP Postfix (Debian)
...,...,...,...,...,...,...,...
4985,Basel,bs.ch,mail.swisscom.com,NaN,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH,220 mail.swisscom.com ESMTP Service Swisscom (...
4986,Lausanne,lausanne.ch,mxvdl.lausanne.ch,NaN,SIL-CITYCABLE-AS - Services Industriels de Lau...,CH,"220 lsamgwp03.lausanne.ch ESMTP Smtpd; Mon, 8 ..."
4987,Schaffhausen,stadt-schaffhausen.ch,miraculix2.ksd.ch,NaN,"KSD - Informatik Schaffhausen (ITSH), CH",CH,554-miraculix2.ksd.ch
4988,Bülach,buelach.ch,buelach-ch.mail.protection.outlook.com,Microsoft 365,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,220 AMS1EPF0000003F.mail.protection.outlook.co...


In [31]:
from collections import Counter
domain_mx = pd.merge(domain[["name", "website_domain"]], mx, on="website_domain")

def count_mx_parts(df):
    counter = Counter()

    for mx in df["smtp_banner"].dropna():
        parts = mx.split(".")
        counter.update(parts)

    return (
        pd.DataFrame(
            counter.items(),
            columns=["part", "count"]
        )
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

count_mx_parts(domain_smtp)

,part,count
0,de ESMTP Postfix,914
1,220 mail,478
2,mail,375
3,outlook,351
4,protection,350
...,...,...
2862,220 lsamgwp03,1
2863,220-c2281,1
2864,hs-nb,1
2865,com ESMTP 41be03b00d2f7-c85df0e5380si25752674a12,1


In [35]:
import re

BANNER_RULES = [
    # Microsoft / Outlook / Exchange Online
    (re.compile(r"outlook|protection\.outlook|mail\.protection\.outlook"), "Microsoft 365"),

    # Hornetsecurity / Antispameurope
    (re.compile(r"hornetsecurity|antispameurope"), "Hornetsecurity"),

    # Postfix (MTA, self-hosted)
    (re.compile(r"\bpostfix\b"), "Self Hosted"),

    # KVNB / KVNBW (unscharf, regional IT)
    (re.compile(r"kvnbw|kvnb"), "KVNBW"),

    # # Generic SMTP / Relay Hinweise
    # (re.compile(r"\besmtp\b|smtp relay|mail relay"), "Generischer SMTP Relay"),

    # Sophos Mail Gateway
    (re.compile(r"sophos|mailgateway.*sophos"), "Sophos Mail Gateway"),

    # # Hydra (unspezifisch, oft Tool/Appliance)
    # (re.compile(r"\bhydra\b"), "Hydra (unsicher: Tool oder interne Appliance)"),

    # Mimecast (sehr häufig Enterprise Mail Security)
    (re.compile(r"mimecast"), "Mimecast Email Security"),

    # Your-server (typisch Hosting / VPS / Hetzner-ähnliche Umgebungen)
    (re.compile(r"your[- ]?server"), "Hetzner"),
]


def identify_provider(mx_domain: str) -> str | None:
    mx_domain = mx_domain.lower()

    for pattern, provider in BANNER_RULES:
        if pattern.search(mx_domain):
            return provider

    return None

domain_smtp["smtp_provider"] = (
    domain_smtp["smtp_banner"]
    .fillna("")
    .apply(identify_provider)
)
domain_smtp.drop(columns="smtp_banner")#.drop_duplicates(subset="name", inplace=True)

,name,website_domain,mx_domain,mx_provider,owner,country,smtp_provider
0,Duale Hochschule Baden-Württemberg Lörrach,dhbw-loerrach.de,mx2.belwue.de,BelWü,"BELWUE - Universitaet Stuttgart, DE",DE,Self Hosted
1,Burg Giebichenstein Kunsthochschule Halle,burg-halle.de,burg-halle.de,NaN,"HLKOMM - HL komm Telekommunikations GmbH, DE",DE,NaN
2,BSP Business & Law School,businessschool-berlin.de,mx-hh.mail-bz.de,NaN,"VERSATEL - 1&1 Versatel GmbH, DE",DE,NaN
3,CVJM-Hochschule,cvjm-hochschule.de,mx1.serve-me.de,NaN,"HETZNER-AS - Hetzner Online GmbH, DE",DE,NaN
4,Technische Universität Hamburg,tuhh.de,smtp3.rz.tu-harburg.de,NaN,DFN - Verein zur Foerderung eines Deutschen Fo...,DE,Self Hosted
...,...,...,...,...,...,...,...
4985,Basel,bs.ch,mail.swisscom.com,NaN,"SWISSCOM - Swisscom (Schweiz) AG, CH",CH,NaN
4986,Lausanne,lausanne.ch,mxvdl.lausanne.ch,NaN,SIL-CITYCABLE-AS - Services Industriels de Lau...,CH,NaN
4987,Schaffhausen,stadt-schaffhausen.ch,miraculix2.ksd.ch,NaN,"KSD - Informatik Schaffhausen (ITSH), CH",CH,NaN
4988,Bülach,buelach.ch,buelach-ch.mail.protection.outlook.com,Microsoft 365,MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpor...,GB,Microsoft 365


In [51]:
def coverage_table(df):
    total = len(df)

    mx = df["mx_provider"].notna()
    owner = df["owner"].notna()
    smtp = df["smtp_provider"].notna()

    mx_only = mx.sum()

    mx_or_owner = mx | (~mx & owner)
    owner_added = mx_or_owner.sum()

    mx_owner_or_smtp = mx_or_owner | (~mx_or_owner & smtp)
    smtp_added = mx_owner_or_smtp.sum()

    print()
    print(f"MX only: {mx.mean() * 100:.3f}%")
    print(f"ASN added: {(owner_added - mx_only) / total * 100:.3f}%")
    print(f"SMTP added: {(smtp_added - owner_added) / total * 100:.3f}%")
    print(f"Final coverage: {smtp_added / total * 100:.3f}%")

coverage_table(domain_smtp)


MX only: 11.663%
ASN added: 72.445%
SMTP added: 0.000%
Final coverage: 84.108%


In [50]:
len(domain_smtp[(domain_smtp["mx_provider"].notna() | domain_smtp["owner"].notna() | domain_smtp["smtp_provider"].notna())]) / len(domain_smtp)

0.8410821643286573

In [11]:
class Test:
    pass

test = Test()

print(type(test).__name__)

Test


In [4]:
import pandas as pd
df = pd.read_csv("/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/mail_server_scan_results.csv")
df

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,...,latitude,timestamp,email_mail_server,email_mail_port,email_mail_protocol,email_mail_banner,website_domain_mail_server,website_domain_mail_port,website_domain_mail_protocol,website_domain_mail_banner
0,f604925d-96f6-4313-843c-f92da890e844,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,Deutschland,http://www.cvjm-hochschule.de/,cvjm.de,university,cvjm-hochschule.de,...,51.303670,2026-05-31 20:08:03.862087,mail.cvjm.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,NaN,NaN,NaN,NaN
1,fec9310f-4548-448c-83c6-b9f2be1e42bd,http://www.wikidata.org/entity/Q107440148,BSBI School of Business and Innovation,Berlin,Berlin,Deutschland,https://www.berlinsbi.com/,berlinsbi.com,university,berlinsbi.com,...,NaN,2026-05-31 20:08:03.862087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,70ea6341-72c6-4496-bbe7-a1e1273cc7dd,http://www.wikidata.org/entity/Q110489587,University of Labour,Frankfurt am Main,Hessen,Deutschland,https://www.university-of-labour.de/,academy-of-labour.de,university,university-of-labour.de,...,50.130408,2026-05-31 20:08:03.862087,imap.academy-of-labour.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,mail.university-of-labour.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
3,50c0ecbf-a773-4804-bd6e-69d2e85a717c,http://www.wikidata.org/entity/Q110624130,Charlotte Fresenius Hochschule,München,Hessen,Deutschland,https://www.charlotte-fresenius-uni.de/,hs-fresenius.de,university,charlotte-fresenius-uni.de,...,NaN,2026-05-31 20:08:03.862087,mail.hs-fresenius.de,587.0,SMTP-STARTTLS,220 srv-maildovecotdirector01 Dovecot (Ubuntu)...,NaN,NaN,NaN,NaN
4,c15f7a86-68d6-4ea2-a88c-a3390c8244b3,http://www.wikidata.org/entity/Q1109275,Collegium Orientale,NaN,NaN,Deutschland,http://www.collegium-orientale.de,bistum-eichstaett.de,university,collegium-orientale.de,...,48.890600,2026-05-31 20:08:03.862087,mail.bistum-eichstaett.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,webmail.collegium-orientale.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2410,55438d6a-e7fa-4407-8bd2-48dcbc17dbc9,http://www.wikidata.org/entity/Q69734,Bassersdorf,Bassersdorf,NaN,Schweiz,https://www.bassersdorf.ch,bassersdorf.ch,city,bassersdorf.ch,...,47.443056,2026-05-31 20:08:03.862087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2411,70cec793-a7c4-455c-b6b5-67b7f267f74e,http://www.wikidata.org/entity/Q69752,Rüti,Rüti,NaN,Schweiz,https://www.rueti.ch,rueti.ch,city,rueti.ch,...,47.261389,2026-05-31 20:08:03.862087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2412,c165c74b-29f7-40eb-9c7f-943091fa8433,http://www.wikidata.org/entity/Q69783,Münsingen,Münsingen,NaN,Schweiz,https://www.muensingen.ch,muensingen.ch,city,muensingen.ch,...,46.872980,2026-05-31 20:08:03.862087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2413,fb158c02-0df9-4ad8-816c-6f77c0fabecc,http://www.wikidata.org/entity/Q70,Bern,Bern,NaN,Schweiz,https://www.bern.ch/it,bern.ch,city,bern.ch,...,46.947980,2026-05-31 20:08:03.862087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df[df['email_mail_protocol'] == "IMAP"]

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,...,latitude,timestamp,email_mail_server,email_mail_port,email_mail_protocol,email_mail_banner,website_domain_mail_server,website_domain_mail_port,website_domain_mail_protocol,website_domain_mail_banner
0,f604925d-96f6-4313-843c-f92da890e844,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,Deutschland,http://www.cvjm-hochschule.de/,cvjm.de,university,cvjm-hochschule.de,...,51.303670,2026-05-31 20:08:03.862087,mail.cvjm.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,NaN,NaN,NaN,NaN
2,70ea6341-72c6-4496-bbe7-a1e1273cc7dd,http://www.wikidata.org/entity/Q110489587,University of Labour,Frankfurt am Main,Hessen,Deutschland,https://www.university-of-labour.de/,academy-of-labour.de,university,university-of-labour.de,...,50.130408,2026-05-31 20:08:03.862087,imap.academy-of-labour.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,mail.university-of-labour.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
4,c15f7a86-68d6-4ea2-a88c-a3390c8244b3,http://www.wikidata.org/entity/Q1109275,Collegium Orientale,NaN,NaN,Deutschland,http://www.collegium-orientale.de,bistum-eichstaett.de,university,collegium-orientale.de,...,48.890600,2026-05-31 20:08:03.862087,mail.bistum-eichstaett.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,webmail.collegium-orientale.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
5,2d4b3edf-a951-4f21-ad46-a0c8602e874c,http://www.wikidata.org/entity/Q1109299,Collegium Willibaldinum,Eichstätt,Bayern,Deutschland,https://www.priesterseminar-eichstaett.de/star...,bistum-eichstaett.de,university,priesterseminar-eichstaett.de,...,48.890658,2026-05-31 20:08:03.862087,mail.bistum-eichstaett.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,NaN,NaN,NaN,NaN
6,fa4c1304-92be-4ebc-b247-136e38e94cad,http://www.wikidata.org/entity/Q113633597,Universität Koblenz,Koblenz,Rheinland-Pfalz,Deutschland,https://www.uni-koblenz.de/de,uni-koblenz.de,university,uni-koblenz.de,...,50.363611,2026-05-31 20:08:03.862087,imap.uni-koblenz.de,143.0,IMAP,* OK IMAP4 ready,smtp.uni-koblenz.de,25.0,SMTP,220 mailproxy.uni-koblenz.de ESMTP Postfix (Ub...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2319,ea79a8b4-1f0a-4d65-a0a4-c0bfd97abef8,http://www.wikidata.org/entity/Q7033,Nordhausen,Nordhausen,Thüringen,Deutschland,https://www.nordhausen.de/,nordhausen.de,city,nordhausen.de,...,51.505000,2026-05-31 20:08:03.862087,webmail.nordhausen.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,mail.nordhausen.de,587.0,SMTP-STARTTLS,220 vwp0542.webpack.hosteurope.de ESMTP Host E...
2348,607b62b9-fad7-4159-a880-f2d0013e5ab6,http://www.wikidata.org/entity/Q8240,Stadt Wehlen,Stadt Wehlen,Sachsen,Deutschland,https://www.wehlen-online.de/,wehlen-online.de,city,wehlen-online.de,...,50.956944,2026-05-31 20:08:03.862087,webmail.wehlen-online.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,imap.wehlen-online.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...
2353,7e6534ed-1225-452c-af7a-01a268f8a8e7,http://www.wikidata.org/entity/Q8714,Großenhain,Großenhain,Sachsen,Deutschland,https://www.grossenhain.de/,grossenhain.de,city,grossenhain.de,...,51.291944,2026-05-31 20:08:03.862087,mail.grossenhain.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 LOGIN-REFERRALS ID ...,imap.grossenhain.de,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 LOGIN-REFERRALS ID ...
2367,6f07fd32-4ac5-4804-a443-bff9ffbe4cbf,http://www.wikidata.org/entity/Q259214,Rust,Rust,NaN,Österreich,https://www.freistadt-rust.at/,freistadt-rust.at,city,freistadt-rust.at,...,47.800833,2026-05-31 20:08:03.862087,mail.freistadt-rust.at,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...,mail.freistadt-rust.at,143.0,IMAP,* OK [CAPABILITY IMAP4rev1 SASL-IR LOGIN-REFER...


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
        "D:\\Projekte\\S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool\\scanner\\database\\raw_data.db"
    )

df = pd.read_sql_query("SELECT * FROM bronze_table", conn)

import json

with open(r"D:\Projekte\S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool\scanner\cache_dump.json") as f:
    cache_dump = json.load(f)
cache_dump

df

DatabaseError: Execution failed on sql 'SELECT * FROM bronze_table': no such table: bronze_table

In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(
        "/home/julian/Projects/S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool/scanner/database/raw_data.db"
    )

df = pd.read_sql_query("SELECT * FROM bronze_table", conn)
df

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,longitude,latitude,timestamp
0,8ea3ab84-6409-4a43-b496-039136f960c4,http://www.wikidata.org/entity/Q1001338,Duale Hochschule Baden-Württemberg Lörrach,Lörrach,Baden-Württemberg,Deutschland,http://www.dhbw-loerrach.de/,NaN,university,dhbw-loerrach.de,7.677240,47.617330,2026-05-31 20:08:03.862087
1,53234a52-8bd8-46ba-8c52-493cd32f82e8,http://www.wikidata.org/entity/Q1011953,Burg Giebichenstein Kunsthochschule Halle,Halle (Saale),Sachsen-Anhalt,Deutschland,https://www.burg-halle.de/,NaN,university,burg-halle.de,11.954900,51.502700,2026-05-31 20:08:03.862087
2,3d1a5654-c62d-4274-a07f-8c4c65334c76,http://www.wikidata.org/entity/Q1017599,BSP Business & Law School,Berlin,Berlin,Deutschland,https://www.businessschool-berlin.de/,NaN,university,businessschool-berlin.de,13.054074,52.406958,2026-05-31 20:08:03.862087
3,f604925d-96f6-4313-843c-f92da890e844,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,Deutschland,http://www.cvjm-hochschule.de/,cvjm.de,university,cvjm-hochschule.de,9.414510,51.303670,2026-05-31 20:08:03.862087
4,215a8e0a-2b57-4b2a-9983-004788e8726c,http://www.wikidata.org/entity/Q1060,Technische Universität Hamburg,Hamburg,Hamburg,Deutschland,https://www.tuhh.de/,NaN,university,tuhh.de,9.969506,53.460958,2026-05-31 20:08:03.862087
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5018,da919250-2571-4130-8171-749a1294d6d9,http://www.wikidata.org/entity/Q78,Basel,Basel,NaN,Schweiz,https://www.bs.ch/,NaN,city,bs.ch,7.590555,47.560556,2026-05-31 20:08:03.862087
5019,b09f2231-55c6-43f7-8939-d3633e435593,http://www.wikidata.org/entity/Q807,Lausanne,Lausanne,NaN,Schweiz,https://www.lausanne.ch/,lausanne.ch,city,lausanne.ch,6.633333,46.533333,2026-05-31 20:08:03.862087
5020,171d6745-0367-456b-9185-85a729b13aff,http://www.wikidata.org/entity/Q9009,Schaffhausen,Schaffhausen,NaN,Schweiz,https://www.stadt-schaffhausen.ch/,NaN,city,stadt-schaffhausen.ch,8.633860,47.696530,2026-05-31 20:08:03.862087
5021,7f0bc9e7-86a8-4835-a0a8-861943a80979,http://www.wikidata.org/entity/Q9093,Bülach,Bülach,NaN,Schweiz,https://www.buelach.ch,NaN,city,buelach.ch,8.542222,47.518889,2026-05-31 20:08:03.862087


In [2]:
df["mx"] = None
df["smtp"] = None
df["asn"] = None
df["country"] = None

for index, row in df.iterrows():
    domain = cache_dump.get(row["website_domain"])

    if not domain or "mx" not in domain:
        continue

    df.at[index, "mx"] = domain["mx"]

    smtps = []
    ips = []
    for mx in domain["mx"]:
        if mx and mx in cache_dump:
            if "smtp" in cache_dump[mx]:
                smtps.append(cache_dump[mx]["smtp"])

            if "ip" in cache_dump[mx]:
                ips.extend(cache_dump[mx]["ip"])

    if smtps:
        df.at[index, "smtp"] = smtps

    asns = []
    countries = []
    for ip in ips:
        if ip in cache_dump:
            if "asn" in cache_dump[ip]:
                asns.append(cache_dump[ip]["asn"]["owner"])
                countries.append(cache_dump[ip]["asn"]["country"])

    if asns:
        df.at[index, "asn"] = asns
    if countries:
        df.at[index, "country"] = countries
df

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,longitude,latitude,timestamp,mx,smtp,asn
0,b8381380-f672-4c15-a743-fe0ce1509ffe,http://www.wikidata.org/entity/Q1001338,Duale Hochschule Baden-Württemberg Lörrach,Lörrach,Baden-Württemberg,"[DE, DE, DE, DE, DE, DE, DE, DE, DE, DE, DE, DE]",http://www.dhbw-loerrach.de/,NaN,university,dhbw-loerrach.de,7.677240,47.617330,2026-05-08 15:30:11.685839,"[mx4.belwue.de, mx2.belwue.de, mx1.belwue.de, ...","[220 mail-in04.belwue.de ESMTP Postfix, 220 ma...","[BELWUE BelWue-Koordination, DE, BELWUE BelWue..."
1,a32648ed-7586-4ed4-9946-80adb13ccedb,http://www.wikidata.org/entity/Q1011953,Burg Giebichenstein Kunsthochschule Halle,Halle (Saale),Sachsen-Anhalt,[DE],https://www.burg-halle.de/,NaN,university,burg-halle.de,11.954900,51.502700,2026-05-08 15:30:11.685839,[burg-halle.de],[220 *****************************************...,"[HLKOMM 04107 Leipzig, DE]"
2,cf712634-996a-4f39-8990-a56f8411582e,http://www.wikidata.org/entity/Q1017599,BSP Business & Law School,Berlin,Berlin,"[DE, DE]",https://www.businessschool-berlin.de/,NaN,university,businessschool-berlin.de,13.054074,52.406958,2026-05-08 15:30:11.685839,"[mx-hh.mail-bz.de, mx-ber.mail-bz.de]","[220 mx-hh.mail-bz.de - NoSpamProxy ready, 220...","[VERSATEL, DE, VERSATEL, DE]"
3,bef4aea6-6ff7-4ab9-a562-23c24309d4a0,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,"[DE, DE]",http://www.cvjm-hochschule.de/,NaN,university,cvjm-hochschule.de,9.414510,51.303670,2026-05-08 15:30:11.685839,"[mx2.serve-me.de, mx1.serve-me.de]","[220 mx2.serve-me.de ESMTP, 220 mx1.serve-me.d...","[HETZNER-AS, DE, HETZNER-AS, DE]"
4,76eb7cc0-0321-43a4-a8ee-522dc3172a46,http://www.wikidata.org/entity/Q1060,Technische Universität Hamburg,Hamburg,Hamburg,"[DE, DE, DE, DE, DE, DE]",https://www.tuhh.de/,NaN,university,tuhh.de,9.969506,53.460958,2026-05-08 15:30:11.685839,"[smtp3.rz.tu-harburg.de, smtp4.rz.tu-harburg.d...",[220-smtp3.rz.tu-harburg.de ESMTP Postfix (Deb...,[DFN Verein zur Foerderung eines Deutschen For...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5007,db56a6e0-a38c-4745-aec6-4e2397a0dc43,http://www.wikidata.org/entity/Q78,Basel,Basel,NaN,"[CH, CH, CH, CH]",https://www.bs.ch/,NaN,city,bs.ch,7.590555,47.560556,2026-05-08 15:30:11.685839,"[mail.swisscom.com, mail20.swisscom.com, mail1...",[220 mail.swisscom.com ESMTP Service Swisscom ...,"[SWISSCOM Swisscom Switzerland Ltd, CH, SWISSC..."
5008,41ade22b-becb-4763-b1b0-af2a6b472c78,http://www.wikidata.org/entity/Q807,Lausanne,Lausanne,NaN,[CH],https://www.lausanne.ch/,NaN,city,lausanne.ch,6.633333,46.533333,2026-05-08 15:30:11.685839,[mxvdl.lausanne.ch],"[220 lsamgwp03.lausanne.ch ESMTP Smtpd; Mon, 1...","[SIL-CITYCABLE-AS, CH]"
5009,83f3c00c-5800-489f-8af9-0e6798f22610,http://www.wikidata.org/entity/Q9009,Schaffhausen,Schaffhausen,NaN,"[CH, CH]",https://www.stadt-schaffhausen.ch/,NaN,city,stadt-schaffhausen.ch,8.633860,47.696530,2026-05-08 15:30:11.685839,"[miraculix2.ksd.ch, miraculix1.ksd.ch]","[554-miraculix2.ksd.ch, 554-miraculix1.ksd.ch]","[KSD, CH, KSD, CH]"
5010,aca98be5-3763-4199-9b48-a633e1a03434,http://www.wikidata.org/entity/Q9093,Bülach,Bülach,NaN,"[GB, US, US, GB, US, GB, GB, US, CH]",https://www.buelach.ch,NaN,city,buelach.ch,8.542222,47.518889,2026-05-08 15:30:11.685839,"[buelach-ch.mail.protection.outlook.com, mailp...",[220 DB1PEPF000509ED.mail.protection.outlook.c...,[MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpo...


In [3]:
def clean_list(items):
    result = []

    for item in items:
        # Verschachtelte Listen rekursiv auflösen
        if isinstance(item, list):
            result.extend(clean_list(item))
            continue

        # None entfernen
        if item is None:
            continue

        # Strings trimmen
        if isinstance(item, str):
            item = item.strip()

            # Leere Strings entfernen
            if item == "":
                continue

        result.append(item)

    return result

In [4]:
import re

def extract_smtp_tech(banner):
    """
    Extracts common SMTP technologies from a banner string.
    Returns a list of detected technologies.
    """

    if not banner or not isinstance(banner, str):
        return []

    banner_lower = banner.lower()

    tech_map = {
        "postfix": r"postfix",
        "exim": r"exim",
        "sendmail": r"sendmail",
        "exchange": r"microsoft exchange|exchange",
        "microsoft_smtp": r"microsoft esmtp|microsoft smtp",
        "dovecot": r"dovecot",
        "qmail": r"qmail",
        "openssh_smtp": r"openssh",
        "openssl": r"openssl",
        "starttls": r"starttls",
        "tls": r"\btls\b|ssl",
        "esmtp": r"\besmtp\b",
    }

    detected = []

    for tech, pattern in tech_map.items():
        if re.search(pattern, banner_lower):
            detected.append(tech)

    return detected

df["smtp_software"] = df["smtp"].apply(
    lambda banners: [
        extract_smtp_tech(banner)
        for banner in banners
    ] if isinstance(banners, list) else []
)
df["smtp_software"] = df["smtp_software"].apply(clean_list)
df["smtp_software"] = df["smtp_software"].apply(
    lambda x: list(set(x)) if isinstance(x, list) else x
)
df["country"] = df["country"].apply(
    lambda x: list(set(x)) if isinstance(x, list) else x
)
df["asn"] = df["asn"].apply(
    lambda x: list(set(x)) if isinstance(x, list) else x
)

def split_asn(asn_list):

    if not isinstance(asn_list, list):
        return [], []

    providers = []
    countries = []

    for item in asn_list:

        if not isinstance(item, str):
            continue

        parts = item.split(",")

        provider = parts[0].strip()
        country = parts[-1].strip() if len(parts) > 1 else None

        providers.append(provider)

        if country:
            countries.append(country)

    # unique + Reihenfolge behalten
    providers = list(dict.fromkeys(providers))
    countries = list(dict.fromkeys(countries))

    return providers, country

df[["provider", "provider_country"]] = df["asn"].apply(
    lambda x: pd.Series(split_asn(x))
)
def determine_sovereignty(countries):

    if not isinstance(countries, list) or not countries:
        return "medium"

    unique = set(countries)

    if "US" in unique:
        return "low"

    if unique == {"DE"}:
        return "high"

    return "medium"
df["sovereignty_level"] = df["country"].apply(determine_sovereignty)
def determine_provider_type(providers):

    if not isinstance(providers, list):
        return "self_hosted"

    hyperscaler_keywords = [
        "google",
        "microsoft",
        "amazon",
    ]

    for provider in providers:

        if not isinstance(provider, str):
            continue

        provider_lower = provider.lower()

        if any(keyword in provider_lower for keyword in hyperscaler_keywords):
            return "hyperscaler"

    return "self-hosted"

df["provider_type"] = df["provider"].apply(determine_provider_type)
df

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,...,latitude,timestamp,mx,smtp,asn,smtp_software,provider,provider_country,sovereignty_level,provider_type
0,b8381380-f672-4c15-a743-fe0ce1509ffe,http://www.wikidata.org/entity/Q1001338,Duale Hochschule Baden-Württemberg Lörrach,Lörrach,Baden-Württemberg,[DE],http://www.dhbw-loerrach.de/,NaN,university,dhbw-loerrach.de,...,47.617330,2026-05-08 15:30:11.685839,"[mx4.belwue.de, mx2.belwue.de, mx1.belwue.de, ...","[220 mail-in04.belwue.de ESMTP Postfix, 220 ma...","[BELWUE BelWue-Koordination, DE]","[postfix, esmtp]",[BELWUE BelWue-Koordination],DE,high,self-hosted
1,a32648ed-7586-4ed4-9946-80adb13ccedb,http://www.wikidata.org/entity/Q1011953,Burg Giebichenstein Kunsthochschule Halle,Halle (Saale),Sachsen-Anhalt,[DE],https://www.burg-halle.de/,NaN,university,burg-halle.de,...,51.502700,2026-05-08 15:30:11.685839,[burg-halle.de],[220 *****************************************...,"[HLKOMM 04107 Leipzig, DE]",[],[HLKOMM 04107 Leipzig],DE,high,self-hosted
2,cf712634-996a-4f39-8990-a56f8411582e,http://www.wikidata.org/entity/Q1017599,BSP Business & Law School,Berlin,Berlin,[DE],https://www.businessschool-berlin.de/,NaN,university,businessschool-berlin.de,...,52.406958,2026-05-08 15:30:11.685839,"[mx-hh.mail-bz.de, mx-ber.mail-bz.de]","[220 mx-hh.mail-bz.de - NoSpamProxy ready, 220...","[VERSATEL, DE]",[],[VERSATEL],DE,high,self-hosted
3,bef4aea6-6ff7-4ab9-a562-23c24309d4a0,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,[DE],http://www.cvjm-hochschule.de/,NaN,university,cvjm-hochschule.de,...,51.303670,2026-05-08 15:30:11.685839,"[mx2.serve-me.de, mx1.serve-me.de]","[220 mx2.serve-me.de ESMTP, 220 mx1.serve-me.d...","[HETZNER-AS, DE]",[esmtp],[HETZNER-AS],DE,high,self-hosted
4,76eb7cc0-0321-43a4-a8ee-522dc3172a46,http://www.wikidata.org/entity/Q1060,Technische Universität Hamburg,Hamburg,Hamburg,[DE],https://www.tuhh.de/,NaN,university,tuhh.de,...,53.460958,2026-05-08 15:30:11.685839,"[smtp3.rz.tu-harburg.de, smtp4.rz.tu-harburg.d...",[220-smtp3.rz.tu-harburg.de ESMTP Postfix (Deb...,[DFN Verein zur Foerderung eines Deutschen For...,"[postfix, esmtp]",[DFN Verein zur Foerderung eines Deutschen For...,DE,high,self-hosted
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5007,db56a6e0-a38c-4745-aec6-4e2397a0dc43,http://www.wikidata.org/entity/Q78,Basel,Basel,NaN,[CH],https://www.bs.ch/,NaN,city,bs.ch,...,47.560556,2026-05-08 15:30:11.685839,"[mail.swisscom.com, mail20.swisscom.com, mail1...",[220 mail.swisscom.com ESMTP Service Swisscom ...,"[SWISSCOM Swisscom Switzerland Ltd, CH]",[esmtp],[SWISSCOM Swisscom Switzerland Ltd],CH,medium,self-hosted
5008,41ade22b-becb-4763-b1b0-af2a6b472c78,http://www.wikidata.org/entity/Q807,Lausanne,Lausanne,NaN,[CH],https://www.lausanne.ch/,NaN,city,lausanne.ch,...,46.533333,2026-05-08 15:30:11.685839,[mxvdl.lausanne.ch],"[220 lsamgwp03.lausanne.ch ESMTP Smtpd; Mon, 1...","[SIL-CITYCABLE-AS, CH]",[esmtp],[SIL-CITYCABLE-AS],CH,medium,self-hosted
5009,83f3c00c-5800-489f-8af9-0e6798f22610,http://www.wikidata.org/entity/Q9009,Schaffhausen,Schaffhausen,NaN,[CH],https://www.stadt-schaffhausen.ch/,NaN,city,stadt-schaffhausen.ch,...,47.696530,2026-05-08 15:30:11.685839,"[miraculix2.ksd.ch, miraculix1.ksd.ch]","[554-miraculix2.ksd.ch, 554-miraculix1.ksd.ch]","[KSD, CH]",[],[KSD],CH,medium,self-hosted
5010,aca98be5-3763-4199-9b48-a633e1a03434,http://www.wikidata.org/entity/Q9093,Bülach,Bülach,NaN,"[US, CH, GB]",https://www.buelach.ch,NaN,city,buelach.ch,...,47.518889,2026-05-08 15:30:11.685839,"[buelach-ch.mail.protection.outlook.com, mailp...",[220 DB1PEPF000509ED.mail.protection.outlook.c...,[MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpo...,"[microsoft_smtp, esmtp]",[MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpo...,CH,low,hyperscaler


In [5]:
df_copy = df.copy()

drop = [
    "uuid", "id", "city", "state", "website", "email", "website_domain", "mx", "smtp", "asn", "country"
]

rename = {
    "name":"org",
    "website_domain":"domain",
    "category_tag":"category",
    "longitude":"lon",
    "latitude":"lat",
    "timestamp":"last_checked",
}

df_copy.drop(columns=drop, inplace=True)
df_copy.rename(columns=rename, inplace=True)
df_copy.to_dict(orient="records")

[{'org': 'Duale Hochschule Baden-Württemberg Lörrach',
  'category': 'university',
  'lon': 7.67724,
  'lat': 47.61733,
  'last_checked': '2026-05-08 15:30:11.685839',
  'smtp_software': ['postfix', 'esmtp'],
  'provider': ['BELWUE BelWue-Koordination'],
  'provider_country': 'DE',
  'sovereignty_level': 'high',
  'provider_type': 'self-hosted'},
 {'org': 'Burg Giebichenstein Kunsthochschule Halle',
  'category': 'university',
  'lon': 11.9549,
  'lat': 51.5027,
  'last_checked': '2026-05-08 15:30:11.685839',
  'smtp_software': [],
  'provider': ['HLKOMM 04107 Leipzig'],
  'provider_country': 'DE',
  'sovereignty_level': 'high',
  'provider_type': 'self-hosted'},
 {'org': 'BSP Business & Law School',
  'category': 'university',
  'lon': 13.054074,
  'lat': 52.406958,
  'last_checked': '2026-05-08 15:30:11.685839',
  'smtp_software': [],
  'provider': ['VERSATEL'],
  'provider_country': 'DE',
  'sovereignty_level': 'high',
  'provider_type': 'self-hosted'},
 {'org': 'CVJM-Hochschule',
 

In [6]:
import json

with open("json_dump.json", "w", encoding="utf-8") as f:
    json.dump(df_copy.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

In [7]:
import json

with open(r"D:\Projekte\S.M.A.R.T.-Sovereign_Mail_Assessment_and_Rating_Tool\scanner\cache_dump.json") as f:
    cache_dump = json.load(f)
cache_dump

{'dhbw-loerrach.de': {'errors': {},
  'mx': ['mx4.belwue.de', 'mx2.belwue.de', 'mx1.belwue.de', 'mx3.belwue.de']},
 'burg-halle.de': {'errors': {},
  'mx': ['burg-halle.de'],
  'ip': ['212.122.56.1'],
  'smtp': '220 ****************************************************'},
 'businessschool-berlin.de': {'errors': {},
  'mx': ['mx-hh.mail-bz.de', 'mx-ber.mail-bz.de']},
 'cvjm-hochschule.de': {'errors': {},
  'mx': ['mx2.serve-me.de', 'mx1.serve-me.de']},
 'tuhh.de': {'errors': {},
  'mx': ['smtp3.rz.tu-harburg.de',
   'smtp4.rz.tu-harburg.de',
   'smtp5.rz.tu-harburg.de']},
 'berlinsbi.com': {'errors': {},
  'mx': ['mx1.hc6443-94.iphmx.com', 'mx2.hc6443-94.iphmx.com']},
 'university-of-labour.de': {'errors': {},
  'mx': ['universityoflabour-de02ec.mail.protection.outlook.com']},
 'charlotte-fresenius-uni.de': {'errors': {},
  'mx': ['charlottefreseniusuni-de02c1i.mail.protection.outlook.com']},
 'srh-university.de': {'errors': {},
  'mx': ['srhuniversity-de0i.mail.protection.outlook.com']}

In [8]:
scan_df = pd.DataFrame.from_dict(cache_dump, orient="index")
scan_df

,errors,mx,ip,smtp,asn
dhbw-loerrach.de,{},"[mx4.belwue.de, mx2.belwue.de, mx1.belwue.de, ...",NaN,NaN,NaN
burg-halle.de,{},[burg-halle.de],[212.122.56.1],220 ******************************************...,NaN
businessschool-berlin.de,{},"[mx-hh.mail-bz.de, mx-ber.mail-bz.de]",NaN,NaN,NaN
cvjm-hochschule.de,{},"[mx2.serve-me.de, mx1.serve-me.de]",NaN,NaN,NaN
tuhh.de,{},"[smtp3.rz.tu-harburg.de, smtp4.rz.tu-harburg.d...",NaN,NaN,NaN
...,...,...,...,...,...
85.218.125.235,{},NaN,NaN,NaN,"{'asn': '34781', 'owner': 'SIL-CITYCABLE-AS, C..."
195.14.126.81,{},NaN,NaN,NaN,"{'asn': '50849', 'owner': 'ASN-LUGA, CH', 'pre..."
195.14.126.82,{},NaN,NaN,NaN,"{'asn': '50849', 'owner': 'ASN-LUGA, CH', 'pre..."
193.200.220.149,{},NaN,NaN,NaN,"{'asn': '34781', 'owner': 'SIL-CITYCABLE-AS, C..."


In [9]:
df_mx = pd.merge(df, scan_df["mx"], left_on="website_domain",right_index=True)
df_mx

,uuid,id,name,city,state,country,website,email,category_tag,website_domain,...,timestamp,mx_x,smtp,asn,smtp_software,provider,provider_country,sovereignty_level,provider_type,mx_y
0,b8381380-f672-4c15-a743-fe0ce1509ffe,http://www.wikidata.org/entity/Q1001338,Duale Hochschule Baden-Württemberg Lörrach,Lörrach,Baden-Württemberg,[DE],http://www.dhbw-loerrach.de/,NaN,university,dhbw-loerrach.de,...,2026-05-08 15:30:11.685839,"[mx4.belwue.de, mx2.belwue.de, mx1.belwue.de, ...","[220 mail-in04.belwue.de ESMTP Postfix, 220 ma...","[BELWUE BelWue-Koordination, DE]","[postfix, esmtp]",[BELWUE BelWue-Koordination],DE,high,self-hosted,"[mx4.belwue.de, mx2.belwue.de, mx1.belwue.de, ..."
1,a32648ed-7586-4ed4-9946-80adb13ccedb,http://www.wikidata.org/entity/Q1011953,Burg Giebichenstein Kunsthochschule Halle,Halle (Saale),Sachsen-Anhalt,[DE],https://www.burg-halle.de/,NaN,university,burg-halle.de,...,2026-05-08 15:30:11.685839,[burg-halle.de],[220 *****************************************...,"[HLKOMM 04107 Leipzig, DE]",[],[HLKOMM 04107 Leipzig],DE,high,self-hosted,[burg-halle.de]
2,cf712634-996a-4f39-8990-a56f8411582e,http://www.wikidata.org/entity/Q1017599,BSP Business & Law School,Berlin,Berlin,[DE],https://www.businessschool-berlin.de/,NaN,university,businessschool-berlin.de,...,2026-05-08 15:30:11.685839,"[mx-hh.mail-bz.de, mx-ber.mail-bz.de]","[220 mx-hh.mail-bz.de - NoSpamProxy ready, 220...","[VERSATEL, DE]",[],[VERSATEL],DE,high,self-hosted,"[mx-hh.mail-bz.de, mx-ber.mail-bz.de]"
3,bef4aea6-6ff7-4ab9-a562-23c24309d4a0,http://www.wikidata.org/entity/Q1024571,CVJM-Hochschule,Kassel,Hessen,[DE],http://www.cvjm-hochschule.de/,NaN,university,cvjm-hochschule.de,...,2026-05-08 15:30:11.685839,"[mx2.serve-me.de, mx1.serve-me.de]","[220 mx2.serve-me.de ESMTP, 220 mx1.serve-me.d...","[HETZNER-AS, DE]",[esmtp],[HETZNER-AS],DE,high,self-hosted,"[mx2.serve-me.de, mx1.serve-me.de]"
4,76eb7cc0-0321-43a4-a8ee-522dc3172a46,http://www.wikidata.org/entity/Q1060,Technische Universität Hamburg,Hamburg,Hamburg,[DE],https://www.tuhh.de/,NaN,university,tuhh.de,...,2026-05-08 15:30:11.685839,"[smtp3.rz.tu-harburg.de, smtp4.rz.tu-harburg.d...",[220-smtp3.rz.tu-harburg.de ESMTP Postfix (Deb...,[DFN Verein zur Foerderung eines Deutschen For...,"[postfix, esmtp]",[DFN Verein zur Foerderung eines Deutschen For...,DE,high,self-hosted,"[smtp3.rz.tu-harburg.de, smtp4.rz.tu-harburg.d..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5007,db56a6e0-a38c-4745-aec6-4e2397a0dc43,http://www.wikidata.org/entity/Q78,Basel,Basel,NaN,[CH],https://www.bs.ch/,NaN,city,bs.ch,...,2026-05-08 15:30:11.685839,"[mail.swisscom.com, mail20.swisscom.com, mail1...",[220 mail.swisscom.com ESMTP Service Swisscom ...,"[SWISSCOM Swisscom Switzerland Ltd, CH]",[esmtp],[SWISSCOM Swisscom Switzerland Ltd],CH,medium,self-hosted,"[mail.swisscom.com, mail20.swisscom.com, mail1..."
5008,41ade22b-becb-4763-b1b0-af2a6b472c78,http://www.wikidata.org/entity/Q807,Lausanne,Lausanne,NaN,[CH],https://www.lausanne.ch/,NaN,city,lausanne.ch,...,2026-05-08 15:30:11.685839,[mxvdl.lausanne.ch],"[220 lsamgwp03.lausanne.ch ESMTP Smtpd; Mon, 1...","[SIL-CITYCABLE-AS, CH]",[esmtp],[SIL-CITYCABLE-AS],CH,medium,self-hosted,[mxvdl.lausanne.ch]
5009,83f3c00c-5800-489f-8af9-0e6798f22610,http://www.wikidata.org/entity/Q9009,Schaffhausen,Schaffhausen,NaN,[CH],https://www.stadt-schaffhausen.ch/,NaN,city,stadt-schaffhausen.ch,...,2026-05-08 15:30:11.685839,"[miraculix2.ksd.ch, miraculix1.ksd.ch]","[554-miraculix2.ksd.ch, 554-miraculix1.ksd.ch]","[KSD, CH]",[],[KSD],CH,medium,self-hosted,"[miraculix2.ksd.ch, miraculix1.ksd.ch]"
5010,aca98be5-3763-4199-9b48-a633e1a03434,http://www.wikidata.org/entity/Q9093,Bülach,Bülach,NaN,"[US, CH, GB]",https://www.buelach.ch,NaN,city,buelach.ch,...,2026-05-08 15:30:11.685839,"[buelach-ch.mail.protection.outlook.com, mailp...",[220 DB1PEPF000509ED.mail.protection.outlook.c...,[MICROSOFT-CORP-MSN-AS-BLOCK - Microsoft Corpo...,"[microsoft_smtp, esmtp]"

In [10]:
def map_ips_to_mx(data):
    """
    Mappt IPs zu ihren MX-Records.
    
    Für jeden Eintrag mit MX-Records wird geschaut, ob die MX-Domains
    selbst im Dictionary vorhanden sind, und wenn ja, werden ihre IPs zugeordnet.
    
    Args:
        data: Dictionary mit der beschriebenen Struktur
        
    Returns:
        Dictionary mit Mapping von MX-Domain -> IPs
    """
    mx_to_ip = {}
    
    for domain, info in data.items():
        # Prüfen ob MX-Records vorhanden sind
        if "mx" not in info:
            continue
            
        # Durch alle MX-Records iterieren
        for mx_domain in info["mx"]:
            # Prüfen ob der MX-Domain selbst im Datensatz vorhanden ist
            if mx_domain in data and "ip" in data[mx_domain]:
                # IPs des MX-Servers zuordnen
                mx_to_ip[mx_domain] = data[mx_domain]["ip"]
    
    return mx_to_ip


# Erweiterte Version: IPs direkt in die Struktur einfügen
def enrich_mx_with_ips(data):
    """
    Reichert MX-Records mit ihren IPs an.
    
    Ersetzt MX-Einträge durch Dictionaries mit Domain und zugehörigen IPs.
    
    Args:
        data: Dictionary mit der beschriebenen Struktur
        
    Returns:
        Modifiziertes Dictionary (in-place Änderung)
    """
    result = data.copy()
    
    for domain, info in result.items():
        if "mx" not in info:
            continue
            
        # Neue MX-Liste mit IPs erstellen
        enriched_mx = []
        for mx_domain in info["mx"]:
            mx_entry = {"domain": mx_domain, "ips": []}
            
            # IPs hinzufügen falls vorhanden
            if mx_domain in result and "ip" in result[mx_domain]:
                mx_entry["ips"] = result[mx_domain]["ip"]
            
            enriched_mx.append(mx_entry)
        
        result[domain]["mx"] = enriched_mx
    
    return result


# Verwendung:
mx_to_ip_mapping = map_ips_to_mx(cache_dump)
# Oder:
enriched_data = enrich_mx_with_ips(cache_dump)
enriched_data

{'dhbw-loerrach.de': {'errors': {},
  'mx': [{'domain': 'mx4.belwue.de',
    'ips': ['2001:7c0:0:76::4',
     '2001:7c0:0:76::3',
     '129.143.76.4',
     '129.143.76.3']},
   {'domain': 'mx2.belwue.de', 'ips': ['129.143.76.2', '2001:7c0:0:76::2']},
   {'domain': 'mx1.belwue.de',
    'ips': ['2001:7c0:0:76::1',
     '129.143.76.2',
     '2001:7c0:0:76::2',
     '129.143.76.1']},
   {'domain': 'mx3.belwue.de', 'ips': ['2001:7c0:0:76::3', '129.143.76.3']}]},
 'burg-halle.de': {'errors': {},
  'mx': [{'domain': 'burg-halle.de', 'ips': ['212.122.56.1']}],
  'ip': ['212.122.56.1'],
  'smtp': '220 ****************************************************'},
 'businessschool-berlin.de': {'errors': {},
  'mx': [{'domain': 'mx-hh.mail-bz.de', 'ips': ['83.135.251.162']},
   {'domain': 'mx-ber.mail-bz.de', 'ips': ['62.214.51.13']}]},
 'cvjm-hochschule.de': {'errors': {},
  'mx': [{'domain': 'mx2.serve-me.de', 'ips': ['88.99.236.4']},
   {'domain': 'mx1.serve-me.de', 'ips': ['5.9.228.145']}]},
 'tuhh.

In [11]:



def df_to_json(df: pd.DataFrame, path: str):
    rename_cols = {
        "name":"org",
        "website_domain":"domain",
        "category_tag":"category",
        "longitude":"lon",
        "latitude":"lat",
    }

    df.rename(columns=rename_cols, inplace=True)
    return df.to_json(orient="records")

df_to_json(df.sample(20), "")

'[{"uuid":"8774c70d-6417-4466-953a-e471022a85cf","id":"http:\\/\\/www.wikidata.org\\/entity\\/Q480646","org":"Amtsgericht D\\u00fclmen","city":"D\\u00fclmen","state":"Nordrhein-Westfalen","country":["DE"],"website":"http:\\/\\/www.ag-duelmen.nrw.de\\/","email":null,"category":"courthouse","domain":"ag-duelmen.nrw.de","lon":7.27864,"lat":51.8321,"timestamp":"2026-05-08 15:30:11.685839","mx":["relay7m.it.nrw.de","relay7v.it.nrw.de"],"smtp":["220 relay7m.it.nrw.de ESMTP Postfix","220 relay7v.it.nrw.de ESMTP Postfix"],"asn":["IT-NRW, DE"],"smtp_software":["postfix","esmtp"],"provider":["IT-NRW"],"provider_country":"DE","sovereignty_level":"high","provider_type":"self-hosted"},{"uuid":"96f3fbe2-9790-4545-ba6b-bbb50c847d4c","id":"http:\\/\\/www.wikidata.org\\/entity\\/Q572512","org":"Teltow","city":"Teltow","state":"Brandenburg","country":["DE"],"website":"https:\\/\\/www.teltow.de\\/","email":null,"category":"city","domain":"teltow.de","lon":13.270556,"lat":52.402222,"timestamp":"2026-05-08